# 03 — Prophet `changepoint_prior_scale` Tuning

**Goal:** find the `changepoint_prior_scale` value that minimizes forecast
error for each asset class (`equity`, `etf`, `crypto`), via Prophet's own
rolling-origin cross-validation, rather than relying on the library
default (0.05) uniformly across assets with very different trend
behavior.

**Why per asset class, not one global value:** `changepoint_prior_scale`
controls how flexible Prophet's piecewise-linear trend is allowed to be
at changepoints — it's a regularization strength, not a direct
volatility measure. Prophet's own diagnostics documentation frames this
as dataset-specific and recommends tuning it via cross-validation rather
than trusting the default across dissimilar series. The hypothesis here:
assets with sharper, more frequent trend regime changes (crypto) need
more trend flexibility to avoid underfitting real shifts, while steadier
assets (e.g. `TLT`) risk the opposite failure — treating noise as trend
if given too much flexibility. That's a hypothesis to confirm against
this data, not something to assume — if a ticker breaks the expected
ordering, that's worth writing down in the conclusion below, not
smoothing over.

**Output of this notebook:** a value per asset class to paste into
`config.CHANGEPOINT_PRIOR_SCALE_BY_ASSET_CLASS`, replacing the current
placeholder (all three set to 0.05).

In [ ]:
import sys
sys.path.insert(0, "..")  # project root, so `import config` resolves

import warnings
import logging
import itertools

import numpy as np
import pandas as pd
import sqlite3
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

from config import DB_PATH, ASSET_CLASS

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")

pd.set_option("display.precision", 5)

## 1. Pick representative tickers per asset class

Running the full CV grid against every ticker in every class is
expensive (each `changepoint_prior_scale` candidate re-fits Prophet
across multiple rolling folds). Two tickers per class keeps this
tractable while still covering some within-class variation — e.g. a
mega-cap tech equity vs a financial (`AAPL` vs `JPM`), or the two
different macro-driven ETFs (`TLT` vs `GLD`).

**Extend `REPRESENTATIVE_TICKERS` below to include more, or all, tickers
in a class if you want more rigor before finalizing values** — the loop
below already handles however many are listed per class.

In [ ]:
REPRESENTATIVE_TICKERS = {
    "equity": ["AAPL", "JPM"],
    "etf": ["SPY", "TLT"],
    "crypto": ["BTC-USD", "ETH-USD"],
}

# Sanity check against config.ASSET_CLASS so this notebook can't silently
# drift out of sync with config.py's actual ticker->class mapping.
for asset_class, tickers in REPRESENTATIVE_TICKERS.items():
    for t in tickers:
        assert ASSET_CLASS.get(t) == asset_class, (
            f"{t} is listed under '{asset_class}' here but config.ASSET_CLASS "
            f"says '{ASSET_CLASS.get(t)}' — update one or the other."
        )
print("Representative tickers OK, consistent with config.ASSET_CLASS")

## 2. Load price history for each representative ticker

In [ ]:
def load_price_series(ticker: str, db_path: str = DB_PATH) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    df = pd.read_sql(
        "SELECT date AS ds, close AS y FROM raw_prices WHERE ticker = ? ORDER BY date",
        conn, params=(ticker,),
    )
    conn.close()
    df["ds"] = pd.to_datetime(df["ds"])
    return df

series_by_ticker = {}
for asset_class, tickers in REPRESENTATIVE_TICKERS.items():
    for t in tickers:
        df = load_price_series(t)
        series_by_ticker[t] = df
        print(f"{t:10s} ({asset_class:6s}): {len(df):4d} rows, "
              f"{df['ds'].min().date()} -> {df['ds'].max().date()}")

## 3. CV grid search

For each candidate `changepoint_prior_scale`, fit with Prophet's
rolling-origin cross-validation and score with RMSE via
`performance_metrics`. Grid is on a log scale per Prophet's own tuning
guidance.

- `initial`: minimum training window before the first fold
- `period`: spacing between fold cutoffs
- `horizon`: how far forward each fold forecasts — matches
  `FORECAST_HORIZON_DAYS` (30d) since that's what we actually care about
  getting right in production

In [ ]:
CANDIDATE_VALUES = [0.001, 0.01, 0.05, 0.1, 0.5]

CV_INITIAL = "730 days"
CV_PERIOD = "90 days"
CV_HORIZON = "30 days"


def score_changepoint_prior_scale(df: pd.DataFrame, freq: str, cps: float) -> float:
    '''Fit + CV a single ticker at a single changepoint_prior_scale,
    return mean RMSE across folds.'''
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=cps,
        interval_width=0.95,
    )
    model.add_country_holidays(country_name="US")
    model.fit(df[["ds", "y"]])

    df_cv = cross_validation(
        model, initial=CV_INITIAL, period=CV_PERIOD, horizon=CV_HORIZON,
        disable_tqdm=True,
    )
    df_perf = performance_metrics(df_cv)
    return df_perf["rmse"].mean()

In [ ]:
results = []

for asset_class, tickers in REPRESENTATIVE_TICKERS.items():
    freq = "D" if asset_class == "crypto" else "B"
    for ticker in tickers:
        df = series_by_ticker[ticker]
        for cps in CANDIDATE_VALUES:
            rmse = score_changepoint_prior_scale(df, freq, cps)
            results.append({
                "asset_class": asset_class, "ticker": ticker,
                "changepoint_prior_scale": cps, "rmse": rmse,
            })
            print(f"{ticker:10s} cps={cps:<6} rmse={rmse:.4f}")

results_df = pd.DataFrame(results)

## 4. Aggregate by asset class and pick the winner

In [ ]:
summary = (
    results_df
    .groupby(["asset_class", "changepoint_prior_scale"])["rmse"]
    .mean()
    .reset_index()
    .sort_values(["asset_class", "rmse"])
)
summary

In [ ]:
best_by_class = (
    summary.loc[summary.groupby("asset_class")["rmse"].idxmin()]
    .set_index("asset_class")["changepoint_prior_scale"]
    .to_dict()
)
print("Best changepoint_prior_scale per asset class (by mean RMSE):")
for asset_class, cps in best_by_class.items():
    print(f"  {asset_class:8s}: {cps}")

print()
print("Paste into config.py:")
print("CHANGEPOINT_PRIOR_SCALE_BY_ASSET_CLASS = {")
for asset_class, cps in best_by_class.items():
    print(f'    "{asset_class}": {cps},')
print("}")

## 5. Rationale (fill in after running the above)

*Replace this cell with your actual findings once the grid above has run
against the real database.*

- **Chosen values:** `equity=___`, `etf=___`, `crypto=___`
- **Did the results match the hypothesis** (crypto needs more trend
  flexibility than equities/ETFs)? ___
- **Any ticker that broke the expected ordering, and a plausible reason
  why:** ___
- **Anything that would change with more representative tickers per
  class, if you extended `REPRESENTATIVE_TICKERS` above:** ___